# Real-Time Audio Extraction and Speech-to-Text Transcription

Notebook này cho phép bạn thực hiện quy trình:
1. **Chọn Video**: Qua giao diện tương tác (hoặc nhập đường dẫn trực tiếp).
2. **Tách Âm Thanh**: Sử dụng `FFmpeg` tách thành file WAV 16kHz mono.
3. **Dịch Giọng Nói**: Sử dụng mô hình `Whisper` dịch âm thanh theo thời gian thực (từng phân đoạn).

**LƯU Ý QUAN TRỌNG VỀ GIAO DIỆN (IPYWIDGETS)**:
* Nếu bạn chạy trên VS Code hoặc một số trình duyệt và **không thấy giao diện nút bấm tương tác (ipywidgets) xuất hiện**, vui lòng cuộn xuống phần **CHẠY THỦ CÔNG / CẤU HÌNH THỦ CÔNG** ở cuối mỗi cell để điền cấu hình bằng code và bỏ comment dòng chạy trực tiếp.

In [24]:
# 1. KIỂM TRA VÀ CÀI ĐẶT THƯ VIỆN
import sys
import subprocess

def check_and_install(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
        print(f"✓ {import_name} đã sẵn sàng.")
    except ImportError:
        print(f"⚠️ Không tìm thấy {import_name}. Đang tiến hành cài đặt...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"✓ Cài đặt thành công {import_name}.")

check_and_install("ipywidgets")
check_and_install("transformers")
check_and_install("soundfile")
check_and_install("librosa")
check_and_install("torch")
check_and_install("tqdm")
print("Môi trường đã sẵn sàng!")

✓ ipywidgets đã sẵn sàng.
✓ transformers đã sẵn sàng.
✓ soundfile đã sẵn sàng.
✓ librosa đã sẵn sàng.
✓ torch đã sẵn sàng.
✓ tqdm đã sẵn sàng.
Môi trường đã sẵn sàng!


In [25]:
# 2. GIAO DIỆN CHỌN VIDEO
import os
import glob
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# =====================================================================
# CẤU HÌNH THỦ CÔNG (Sử dụng nếu giao diện tương tác không hiển thị)
# =====================================================================
# Bạn có thể nhập trực tiếp đường dẫn video của bạn vào đây:
MANUAL_VIDEO_PATH = None
# Ví dụ:
# MANUAL_VIDEO_PATH = os.path.abspath("demo_data/videos/synthetic_lecture.mp4")


# Xác định đường dẫn tương đối tới các thư mục chứa video demo
current_dir = os.getcwd()
DEMO_VIDEO_DIR = os.path.join(current_dir, "demo_data", "videos")

demo_video_options = {}
if os.path.exists(DEMO_VIDEO_DIR):
    videos = glob.glob(os.path.join(DEMO_VIDEO_DIR, "*.mp4")) + glob.glob(os.path.join(DEMO_VIDEO_DIR, "*.mkv"))
    demo_video_options = {os.path.basename(v): os.path.abspath(v) for v in videos}

selector_mode = widgets.RadioButtons(
    options=[
        ('Chọn video demo sẵn có', 'demo'),
        ('Nhập đường dẫn file thủ công', 'path'),
        ('Mở cửa sổ chọn file Windows Explorer', 'explorer'),
        ('Tải video lên từ trình duyệt (dung lượng thấp)', 'upload')
    ],
    value='demo' if demo_video_options else 'path',
    description='Phương thức:',
    layout=widgets.Layout(width='550px')
)

demo_dropdown = widgets.Dropdown(
    options=demo_video_options if demo_video_options else {'Không tìm thấy video demo': ''},
    description='Video demo:',
    layout=widgets.Layout(width='500px')
)

path_input = widgets.Text(
    value='',
    placeholder='Ví dụ: D:\\Lectures\\lecture_01.mp4',
    description='Đường dẫn:',
    layout=widgets.Layout(width='600px')
)

upload_btn = widgets.FileUpload(
    accept='.mp4,.mkv,.avi,.mov',
    multiple=False,
    description='Tải video lên',
    layout=widgets.Layout(width='200px')
)

btn_explorer = widgets.Button(
    description='Mở hộp thoại chọn file...',
    button_style='info',
    icon='folder-open-o',
    layout=widgets.Layout(width='250px')
)

selection_output = widgets.Output()
selected_video_path = ""

def update_ui(*args):
    with selection_output:
        clear_output()
    
    demo_dropdown.layout.display = 'none'
    path_input.layout.display = 'none'
    btn_explorer.layout.display = 'none'
    upload_btn.layout.display = 'none'
    
    if selector_mode.value == 'demo':
        demo_dropdown.layout.display = 'block'
    elif selector_mode.value == 'path':
        path_input.layout.display = 'block'
    elif selector_mode.value == 'explorer':
        btn_explorer.layout.display = 'block'
    elif selector_mode.value == 'upload':
        upload_btn.layout.display = 'block'

def on_explorer_click(b):
    global selected_video_path
    with selection_output:
        clear_output()
        print("Đang mở hộp thoại chọn file...")
        try:
            import tkinter as tk
            from tkinter import filedialog
            root = tk.Tk()
            root.withdraw()
            root.attributes('-topmost', True)
            file_path = filedialog.askopenfilename(
                title="Chọn file video bài giảng",
                filetypes=[("Video files", "*.mp4 *.mkv *.avi *.mov"), ("All files", "*.*")]
            )
            root.destroy()
            if file_path:
                selected_video_path = os.path.abspath(file_path)
                print(f"✓ Đã chọn file: {selected_video_path}")
            else:
                print("Chưa chọn file nào.")
        except Exception as e:
            print(f"❌ Không thể mở hộp thoại chọn file (Lỗi: {e}). Vui lòng nhập đường dẫn thủ công vào MANUAL_VIDEO_PATH.")

def get_selected_video():
    global selected_video_path
    
    # Kiểm tra cấu hình thủ công trước tiên
    if MANUAL_VIDEO_PATH:
        selected_video_path = os.path.abspath(MANUAL_VIDEO_PATH)
        print(f"✓ Sử dụng đường dẫn thủ công: {selected_video_path}")
        return selected_video_path
        
    mode = selector_mode.value
    if mode == 'demo':
        selected_video_path = demo_dropdown.value
    elif mode == 'path':
        selected_video_path = path_input.value
    elif mode == 'explorer':
        pass
    elif mode == 'upload':
        if upload_btn.value:
            uploaded_file = upload_btn.value
            if isinstance(uploaded_file, dict):
                filename = list(uploaded_file.keys())[0]
                content = uploaded_file[filename]['content']
            else:
                file_info = uploaded_file[0]
                filename = file_info['name']
                content = file_info['content']
                
            os.makedirs(DEMO_VIDEO_DIR, exist_ok=True)
            selected_video_path = os.path.abspath(os.path.join(DEMO_VIDEO_DIR, filename))
            with open(selected_video_path, 'wb') as f:
                f.write(content)
            print(f"✓ Đã lưu file tải lên: {selected_video_path}")
            
    if not selected_video_path or not os.path.exists(selected_video_path):
        raise FileNotFoundError(f"Không tồn tại file video tại: {selected_video_path}. Vui lòng điền vào biến MANUAL_VIDEO_PATH ở trên.")
    return selected_video_path

btn_explorer.on_click(on_explorer_click)
selector_mode.observe(update_ui, 'value')
update_ui()

display(widgets.VBox([
    widgets.HTML("<h2 style='color: #2b5797;'>1. Giao diện Chọn Video</h2>"),
    selector_mode,
    demo_dropdown,
    path_input,
    btn_explorer,
    upload_btn,
    widgets.HTML("<br>"),
    selection_output
]))

In [ ]:
# 3. TÁCH ÂM THANH SỬ DỤNG FFMEG CỤC BỘ
import time
import subprocess

def extract_audio_from_video(video_path):
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Không tìm thấy file: {video_path}")
        
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    audio_dir = os.path.join(os.getcwd(), "demo_data", "audio")
    os.makedirs(audio_dir, exist_ok=True)
    audio_output = os.path.join(audio_dir, f"{base_name}_extracted.wav")
    
    print(f"Đang tách âm thanh bằng FFmpeg... ")
    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vn",
        "-acodec", "pcm_s16le",
        "-ar", "16000",
        "-ac", "1",
        audio_output
    ]
    
    try:
        start_time = time.time()
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
        duration = time.time() - start_time
        print(f"✓ Đã tách thành công sau {duration:.2f} giây!")
        print(f"Đường dẫn âm thanh WAV: {audio_output}")
        print(f"Kích thước file: {os.path.getsize(audio_output) / (1024 * 1024):.2f} MB")
        return audio_output
    except subprocess.CalledProcessError as e:
        print("❌ Lỗi chạy FFmpeg:")
        print(e.stderr)
        raise e

btn_extract = widgets.Button(
    description='Tách âm thanh từ video',
    button_style='success',
    icon='music',
    layout=widgets.Layout(width='280px')
)
extraction_output = widgets.Output()
extracted_audio_path = ""

def on_extract_click(b):
    global extracted_audio_path
    with extraction_output:
        clear_output()
        try:
            video_path = get_selected_video()
            print(f"Video nguồn: {video_path}")
            extracted_audio_path = extract_audio_from_video(video_path)
        except Exception as e:
            print(f"❌ Lỗi tách âm thanh: {e}")

btn_extract.on_click(on_extract_click)

display(widgets.VBox([
    widgets.HTML("<h2 style='color: #2b5797;'>2. Xử lý Tách Âm Thanh</h2>"),
    btn_extract,
    extraction_output
]))

# =====================================================================
# CHẠY THỦ CÔNG (Nếu không hiển thị widget nút bấm ở trên)
# =====================================================================
# Nếu không có widget hiển thị, hãy bỏ comment 2 dòng dưới đây để chạy trực tiếp:
# video_path = get_selected_video()
# extracted_audio_path = extract_audio_from_video(video_path)

In [27]:
# 4. CẤU HÌNH MÔ HÌNH NHẬN DẠNG WHISPER
import torch
from transformers import pipeline

# =====================================================================
# CẤU HÌNH THỦ CÔNG (Sử dụng nếu giao diện tương tác không hiển thị)
# =====================================================================
MANUAL_MODEL_NAME = "openai/whisper-tiny"  # Các model khác: openai/whisper-base, openai/whisper-small
MANUAL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


model_dropdown = widgets.Dropdown(
    options=[
        ('Whisper Tiny (Rất nhanh, nhẹ, 75MB)', 'openai/whisper-tiny'),
        ('Whisper Base (Cân bằng, 140MB)', 'openai/whisper-base'),
        ('Whisper Small (Độ chính xác cao, 460MB)', 'openai/whisper-small'),
        ('Whisper Medium (Độ chính xác rất cao, 1.5GB)', 'openai/whisper-medium')
    ],
    value='openai/whisper-tiny',
    description='Mô hình:',
    layout=widgets.Layout(width='500px')
)

gpu_available = torch.cuda.is_available()
device_info_html = f"""
<div style=\"padding: 10px; border-radius: 5px; background-color: {'#e6f4ea' if gpu_available else '#feeceb'}; color: {'#137333' if gpu_available else '#c5221f'}; margin: 10px 0; font-family: sans-serif;\">
    <strong>Thiết bị phần cứng phát hiện:</strong> {'Phát hiện GPU CUDA! Sẽ chạy mô hình trên GPU.' if gpu_available else 'Không phát hiện thấy GPU CUDA. Sẽ chạy mô hình trên CPU.'}
</div>
"""

btn_load_model = widgets.Button(
    description='Tải mô hình Whisper vào bộ nhớ',
    button_style='warning',
    icon='download',
    layout=widgets.Layout(width='280px')
)
model_output = widgets.Output()
asr_pipeline = None

def on_load_model_click(b):
    global asr_pipeline
    with model_output:
        clear_output()
        model_name = model_dropdown.value
        device = 0 if torch.cuda.is_available() else -1
        device_str = "GPU (CUDA)" if device == 0 else "CPU"
        
        print(f"Đang khởi tạo ASR pipeline với mô hình: {model_name} trên {device_str}...")
        try:
            start_t = time.time()
            asr_pipeline = pipeline(
                task="automatic-speech-recognition",
                model=model_name,
                device=device
            )
            print(f"✓ Đã tải xong mô hình {model_name} trong {time.time() - start_t:.2f} giây!")
        except Exception as e:
            print(f"❌ Lỗi khi khởi tạo mô hình: {e}")

btn_load_model.on_click(on_load_model_click)

display(widgets.VBox([
    widgets.HTML("<h2 style='color: #2b5797;'>3. Khởi Tạo Mô Hình Nhận Dạng (ASR)</h2>"),
    widgets.HTML(device_info_html),
    model_dropdown,
    btn_load_model,
    model_output
]))

# =====================================================================
# CHẠY THỦ CÔNG (Nếu không hiển thị widget nút bấm ở trên)
# =====================================================================
# Nếu không có widget hiển thị, hãy bỏ comment 4 dòng dưới đây để tải model trực tiếp:
# model_name = MANUAL_MODEL_NAME
# device = 0 if MANUAL_DEVICE == "cuda" else -1
# print(f"Đang tải mô hình {model_name} trên {MANUAL_DEVICE}...")
# asr_pipeline = pipeline(task='automatic-speech-recognition', model=model_name, device=device)

In [28]:
# 5. NHẬN DẠNG GIỌNG NÓI & HIỂN THỊ THỜI GIAN THỰC (Tích hợp VAD)
import soundfile as sf
import librosa
import numpy as np
import torch

# =====================================================================
# CẤU HÌNH THỦ CÔNG (Sử dụng nếu giao diện tương tác không hiển thị)
# =====================================================================
MANUAL_CHUNK_SEC = 5

# Khởi tạo VAD model (Silero VAD) để lọc nhạc nền và tĩnh lặng
print("Đang tải mô hình Voice Activity Detection (Silero VAD)...")
try:
    vad_model, vad_utils = torch.hub.load(
        repo_or_dir='snakers4/silero-vad',
        model='silero_vad',
        trust_repo=True,
        force_reload=False
    )
    get_speech_timestamps = vad_utils[0]
    print("✓ Khởi tạo Silero VAD thành công!")
except Exception as e:
    print(f"⚠️ Không thể tải Silero VAD: {e}. Sẽ sử dụng bộ lọc năng lượng RMS làm dự phòng.")
    vad_model = None
    get_speech_timestamps = None

def has_voice_activity(chunk, sr=16000):
    # 1. Thử dùng Silero VAD chính xác cao
    if vad_model is not None and get_speech_timestamps is not None:
        try:
            chunk_tensor = torch.from_numpy(chunk).float()
            speech_ts = get_speech_timestamps(chunk_tensor, vad_model, sampling_rate=16000)
            return len(speech_ts) > 0
        except Exception:
            pass
    # 2. Fallback sang tính năng lượng RMS của tín hiệu nếu offline/lỗi
    rms = librosa.feature.rms(y=chunk)[0]
    return rms.mean() > 0.012

slider_chunk_size = widgets.IntSlider(
    value=5,
    min=2,
    max=30,
    step=1,
    description='Độ dài đoạn (s):',
    layout=widgets.Layout(width='450px')
)

btn_transcribe = widgets.Button(
    description='Bắt đầu Dịch giọng nói',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='280px')
)
transcribe_output = widgets.Output()

def format_time(seconds):
    mins = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{mins:02d}:{secs:02d}"

def run_transcription(b):
    global extracted_audio_path, asr_pipeline, transcript_list
    
    with transcribe_output:
        clear_output()
        
        if not extracted_audio_path or not os.path.exists(extracted_audio_path):
            print("❌ Lỗi: Bạn chưa thực hiện tách âm thanh hoặc file WAV không tồn tại.")
            return
            
        if asr_pipeline is None:
            print("❌ Lỗi: Mô hình Whisper chưa được tải. Hãy thực hiện bước 3 trước.")
            return
            
        chunk_sec = slider_chunk_size.value
        print(f"Đang chuẩn bị nhận dạng file âm thanh: {extracted_audio_path}")
        print(f"Cấu hình phân đoạn: {chunk_sec} giây / đoạn")
        
        try:
            audio_data, sr = sf.read(extracted_audio_path)
            if len(audio_data.shape) > 1:
                audio_data = audio_data.mean(axis=1)
                
            if sr != 16000:
                print("Đang resample về tần số 16000Hz...")
                audio_data = librosa.resample(audio_data, orig_sr=sr, target_sr=16000)
                sr = 16000
                
            total_duration = len(audio_data) / sr
            print(f"Thời lượng: {format_time(total_duration)} ({total_duration:.2f} giây)")
            
            status_label = widgets.HTML(value="<b>Trạng thái:</b> Đang chuẩn bị...")
            progress_bar = widgets.FloatProgress(
                value=0.0, min=0.0, max=total_duration,
                description='Dịch:', bar_style='info',
                layout=widgets.Layout(width='550px')
            )
            
            transcript_area = widgets.HTML(
                value="""
                <div style="border: 1px solid #ddd; padding: 15px; height: 350px; overflow-y: auto; background-color: #fcfcfc; font-family: sans-serif; border-radius: 6px; box-shadow: inset 0 1px 3px rgba(0,0,0,0.05);">
                    <ul style="list-style-type: none; margin: 0; padding-left: 0; line-height: 1.8; color: #666;">
                        <li><i>Script văn bản dịch sẽ cập nhật tại đây theo thời gian thực...</i></li>
                    </ul>
                </div>
                """
            )
            
            display(status_label)
            display(progress_bar)
            display(transcript_area)
            
            chunk_samples = chunk_sec * sr
            transcript_list = []
            start_perf = time.time()
            
            for i in range(0, len(audio_data), chunk_samples):
                chunk = audio_data[i:i+chunk_samples]
                if len(chunk) < 0.5 * sr:
                    continue
                    
                t_start = i / sr
                t_end = min(len(audio_data), i + chunk_samples) / sr
                
                # CHẠY VOICE ACTIVITY DETECTION (VAD)
                if not has_voice_activity(chunk, sr=sr):
                    status_label.value = f"<b>Trạng thái:</b> Lọc đoạn {format_time(t_start)} - {format_time(t_end)} (Nhạc nền/Im lặng)..."
                    progress_bar.value = t_end
                    transcript_list.append((t_start, t_end, "...", []))
                else:
                    status_label.value = f"<b>Trạng thái:</b> Đang phân tích đoạn {format_time(t_start)} - {format_time(t_end)}..."
                    progress_bar.value = t_end
                    
                    res = asr_pipeline(
                        {"array": chunk, "sampling_rate": sr},
                        return_timestamps="word",
                        generate_kwargs={"task": "transcribe"}
                    )
                    text = res.get("text", "").strip()
                    if not text:
                        text = "..."
                        
                    words = []
                    for c in res.get("chunks", []):
                        w_text = c.get("text", "").strip()
                        ts = c.get("timestamp")
                        if w_text and ts and len(ts) == 2:
                            words.append({
                                "word": w_text,
                                "start": t_start + ts[0],
                                "end": t_start + ts[1]
                            })
                    transcript_list.append((t_start, t_end, text, words))
                
                list_html = ""
                for item in transcript_list:
                    ts, te, txt = item[0], item[1], item[2]
                    if txt == "...":
                        list_html += f"""
                        <li style="margin-bottom: 8px; border-bottom: 1px dashed #eee; padding-bottom: 4px; color: #94a3b8;">
                            <span style="font-weight: bold; font-family: monospace; margin-right: 10px;">[{format_time(ts)} - {format_time(te)}]</span> 
                            <span><i>[Nhạc nền / Im lặng]</i></span>
                        </li>
                        """
                    else:
                        list_html += f"""
                        <li style="margin-bottom: 8px; border-bottom: 1px dashed #eee; padding-bottom: 4px;">
                            <span style="color: #2b5797; font-weight: bold; font-family: monospace; margin-right: 10px;">[{format_time(ts)} - {format_time(te)}]</span> 
                            <span style="color: #333;">{txt}</span>
                        </li>
                        """
                
                transcript_area.value = f"""
                <div style="border: 1px solid #ddd; padding: 15px; height: 350px; overflow-y: auto; background-color: #fcfcfc; font-family: sans-serif; border-radius: 6px; box-shadow: inset 0 1px 3px rgba(0,0,0,0.05);">
                    <ul style="list-style-type: none; margin: 0; padding-left: 0; line-height: 1.8;">
                        {list_html}
                    </ul>
                </div>
                """
                time.sleep(0.05)
                
            status_label.value = f"<b>Trạng thái:</b> ✓ Hoàn thành trong {time.time() - start_perf:.2f} giây!"
            progress_bar.bar_style = 'success'
            
            base_name = os.path.splitext(os.path.basename(extracted_audio_path))[0]
            result_dir = os.path.join(os.getcwd(), "demo_data", "results")
            os.makedirs(result_dir, exist_ok=True)
            txt_output = os.path.join(result_dir, f"{base_name}_transcript.txt")
            
            with open(txt_output, "w", encoding="utf-8") as f:
                for item in transcript_list:
                    ts, te, txt = item[0], item[1], item[2]
                    if txt == "...":
                        f.write(f"[{format_time(ts)} - {format_time(te)}]: [Nhạc nền / Im lặng]\n")
                    else:
                        f.write(f"[{format_time(ts)} - {format_time(te)}]: {txt}\n")
            print(f"\n✓ Đã lưu kết quả tại: {txt_output}")
            
            with open(txt_output, "r", encoding="utf-8") as f:
                full_text = f.read()
                
            txt_box = widgets.Textarea(
                value=full_text,
                description='Sao chép:',
                disabled=False,
                layout=widgets.Layout(width='100%', height='180px')
            )
            display(widgets.HTML("<h4>Sao chép toàn bộ văn bản:</h4>"))
            display(txt_box)
            
        except Exception as e:
            print(f"❌ Đã xảy ra lỗi trong quá trình dịch: {e}")

btn_transcribe.on_click(run_transcription)
display(widgets.VBox([
    widgets.HTML("<h2 style='color: #2b5797;'>4. Tiến Trình Dịch Giọng Nói</h2>"),
    slider_chunk_size,
    btn_transcribe,
    transcribe_output
]))

# =====================================================================
# CHẠY THỦ CÔNG (Nếu không hiển thị widget nút bấm ở trên)
# =====================================================================
# Bỏ comment các dòng dưới đây để thực hiện dịch trực tiếp và hiển thị ra console:
# if 'extracted_audio_path' in globals() and extracted_audio_path and asr_pipeline:
#     audio_data, sr = sf.read(extracted_audio_path)
#     if len(audio_data.shape) > 1: audio_data = audio_data.mean(axis=1)
#     if sr != 16000: audio_data = librosa.resample(audio_data, orig_sr=sr, target_sr=16000); sr = 16000
#     chunk_samples = MANUAL_CHUNK_SEC * sr
#     transcript_list = []
#     print(f'Bắt đầu dịch thủ công (Độ dài đoạn: {MANUAL_CHUNK_SEC}s)...')
#     for idx, i in enumerate(range(0, len(audio_data), chunk_samples)):
#         chunk = audio_data[i:i+chunk_samples]
#         if len(chunk) < 0.5 * sr: continue
#         t_start = i / sr
#         t_end = min(len(audio_data), i + chunk_samples) / sr
#         if not has_voice_activity(chunk, sr=sr):
#             print(f'[{format_time(t_start)} - {format_time(t_end)}]: [Nhạc nền / Im lặng]')
#             transcript_list.append((t_start, t_end, "...", []))
#             continue
#         res = asr_pipeline({'array': chunk, 'sampling_rate': sr}, return_timestamps="word", generate_kwargs={'task': 'transcribe'})
#         text = res.get('text', '').strip()
#         words = []
#         for c in res.get('chunks', []):
#             w_text = c.get('text', '').strip()
#             ts = c.get('timestamp')
#             if w_text and ts and len(ts) == 2:
#                 words.append({'word': w_text, 'start': t_start + ts[0], 'end': t_start + ts[1]})
#         print(f'[{format_time(t_start)} - {format_time(t_end)}]: {text}')
#         transcript_list.append((t_start, t_end, text, words))


Đang tải mô hình Voice Activity Detection (Silero VAD)...
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to C:\Users\hung/.cache\torch\hub\master.zip
✓ Khởi tạo Silero VAD thành công!


# 6. DEMO: PHÁT ÂM THANH & HIGHLIGHT TỪ ĐỒNG BỘ

Phần này chạy một demo tương tác cho phép phát âm thanh từ video đã chọn và highlight từ đang phát âm theo thời gian thực. Bạn cũng có thể click trực tiếp vào bất kỳ từ nào trong văn bản để tua âm thanh tới từ đó.

In [30]:
# 6. DEMO: PHÁT ÂM THANH & HIGHLIGHT TỪ ĐỒNG BỘ
import os
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Kiểm tra dữ liệu từ các bước trước
if 'extracted_audio_path' not in globals() or not extracted_audio_path or not os.path.exists(extracted_audio_path):
    print("❌ Lỗi: Không tìm thấy file âm thanh đã trích xuất. Vui lòng chạy Bước 2 và Bước 3 trước.")
elif 'transcript_list' not in globals() or not transcript_list:
    print("❌ Lỗi: Không tìm thấy kết quả dịch (transcript_list). Vui lòng chạy Bước 5 để dịch âm thanh trước.")
else:
    # 2. Chuyển đổi dữ liệu transcript thực tế thành danh sách từ
    words_data = []
    for item in transcript_list:
        if isinstance(item, dict):
            # Nếu phần tử là Dictionary
            words = item.get('words')
            if words and isinstance(words, list):
                words_data.extend(words)
            else:
                # Fallback: Nội suy tuyến tính cho phân đoạn
                t_start = item.get('start', 0.0)
                t_end = item.get('end', 0.0)
                text = item.get('text', '')
                if text == "...":
                    continue
                words = text.split()
                if not words:
                    continue
                chunk_duration = t_end - t_start
                word_duration = chunk_duration / len(words)
                for idx, w in enumerate(words):
                    words_data.append({
                        "word": w,
                        "start": t_start + idx * word_duration,
                        "end": t_start + (idx + 1) * word_duration
                    })
        elif isinstance(item, (list, tuple)):
            # Nếu phần tử là Tuple/List
            if len(item) >= 4 and isinstance(item[3], list) and item[3]:
                words_data.extend(item[3])
            else:
                # Fallback: Nội suy tuyến tính cho phân đoạn
                t_start, t_end, text = item[0], item[1], item[2]
                if text == "...":
                    continue
                words = text.split()
                if not words:
                    continue
                chunk_duration = t_end - t_start
                word_duration = chunk_duration / len(words)
                for idx, w in enumerate(words):
                    words_data.append({
                        "word": w,
                        "start": t_start + idx * word_duration,
                        "end": t_start + (idx + 1) * word_duration
                    })
            
    # 3. Tính toán đường dẫn tương đối để Jupyter Notebook có thể phát trực tiếp qua trình duyệt
    notebook_dir = os.getcwd()
    rel_audio_path = os.path.relpath(extracted_audio_path, notebook_dir).replace('\\', '/')
    
    # 4. Render HTML/CSS/JS Player
    words_html = []
    for idx, w in enumerate(words_data):
        clean_word = w["word"].strip()
        words_html.append(
            f'<span class="demo-word-span" id="d-word-{idx}" data-start="{w["start"]:.3f}" data-end="{w["end"]:.3f}">{clean_word}</span>'
        )
    words_container_html = " ".join(words_html)

    html_content = f"""
<div class='demo-player-box'>
    <div style='display: flex; align-items: center; justify-content: space-between; margin-bottom: 12px;'>
        <span style='font-weight: bold; color: #2b5797; font-size: 1.1rem; display: flex; align-items: center; gap: 8px;'>
            🔊 Trình phát bài giảng thực tế: {os.path.basename(extracted_audio_path)}
        </span>
        <span style='font-size: 0.85rem; color: #666; font-family: monospace;'>
            Tổng số từ: {len(words_data)}
        </span>
    </div>
    
    <div style='display: flex; justify-content: center; margin-bottom: 18px; background: #f1f5f9; padding: 10px; border-radius: 8px;'>
        <audio id='demo-player-audio' controls src='{rel_audio_path}' style='width: 100%; max-width: 500px;'></audio>
    </div>
    
    <div class='demo-words-container'>
        {words_container_html}
    </div>
    
    <div style='margin-top: 12px; font-size: 0.8rem; color: #64748b; text-align: center;'>
        💡 Hướng dẫn: Nhấp vào nút Play để nghe. Nhấp vào từ bất kỳ để tua âm thanh đến từ đó.
    </div>
</div>

<style>
    .demo-player-box {{
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
        background: #ffffff;
        border: 1px solid #e2e8f0;
        border-radius: 16px;
        padding: 20px;
        box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.05), 0 4px 6px -4px rgba(0, 0, 0, 0.05);
        margin: 15px auto;
        max-width: 700px;
    }}
    .demo-words-container {{
        background: #f8fafc;
        border: 1px solid #e2e8f0;
        border-radius: 12px;
        padding: 20px;
        line-height: 2.2;
        font-size: 1.2rem;
        text-align: justify;
        max-height: 250px;
        overflow-y: auto;
    }}
    .demo-word-span {{
        display: inline-block;
        padding: 1px 6px;
        margin: 2px 2px;
        border-radius: 6px;
        color: #334155;
        cursor: pointer;
        transition: all 0.1s ease-in-out;
        border-bottom: 2px solid transparent;
    }}
    .demo-word-span:hover {{
        background: #e2e8f0;
        color: #0f172a;
    }}
    .demo-word-span.active-word {{
        background: #2563eb !important;
        color: #ffffff !important;
        font-weight: 600;
        transform: translateY(-1px);
        box-shadow: 0 4px 6px -1px rgba(37, 99, 235, 0.2);
    }}
</style>

<script>
    (function() {{
        const audio = document.getElementById('demo-player-audio');
        const spans = document.querySelectorAll('.demo-word-span');
        
        spans.forEach(span => {{
            span.addEventListener('click', function() {{
                const start = parseFloat(this.getAttribute('data-start'));
                audio.currentTime = start;
                if (audio.paused) {{
                    audio.play();
                }}
            }});
        }});
        
        audio.addEventListener('timeupdate', function() {{
            const t = audio.currentTime;
            let activeSpan = null;
            
            spans.forEach(span => {{
                const start = parseFloat(span.getAttribute('data-start'));
                const end = parseFloat(span.getAttribute('data-end'));
                if (t >= start && t <= end) {{
                    span.classList.add('active-word');
                    activeSpan = span;
                }} else {{
                    span.classList.remove('active-word');
                }}
            }});
            
            if (activeSpan) {{
                const container = document.querySelector('.demo-words-container');
                const cTop = container.scrollTop;
                const cBottom = cTop + container.clientHeight;
                const eTop = activeSpan.offsetTop - container.offsetTop;
                const eBottom = eTop + activeSpan.clientHeight;
                
                if (eTop < cTop || eBottom > cBottom) {{
                    container.scrollTo({{
                        top: eTop - container.clientHeight / 2,
                        behavior: 'smooth'
                    }});
                }}
            }}
        }});
    }})();
</script>
"""
    display(HTML(html_content))
